In [2]:
import pandas as pd
import numpy as np
import joblib

df=pd.read_csv(
      "../data/processed/customer_clean.csv"
    )

In [3]:
cluster_df = pd.read_csv(
    "../reports/segmentation/clustered_customers.csv"
)

In [4]:
best_model = joblib.load(
    "../models/best_model.pkl"
)

In [16]:
importance_df = pd.read_csv(
    "../reports/shap/feature_importance.csv"
)

In [5]:
prediction_df=df.drop(
    columns=[
        "customerID",
        "Churn"
    ]
)

In [6]:
churn_probability=best_model.predict_proba(
    prediction_df
)[:,1]

In [7]:
df["Churn Probability"] = churn_probability

In [8]:
def risk_category(prob):

    if prob>=0.70:
        return "High Risk"

    elif prob>=0.40:
        return "Medium Risk"

    else:
        return "Low Risk"


df["Risk Category"]=df["Churn Probability"].apply(risk_category)

In [9]:
df["Risk Category"].value_counts()

Risk Category
Low Risk       4921
Medium Risk    1599
High Risk       523
Name: count, dtype: int64

In [11]:
def recommend(customer):

    if(
        customer["Risk Category"]=="High Risk"
        and customer["MonthlyCharges"]>80
    ):
        return "Offer 20% discount and assign a retention specialist."

    elif(
        customer["Risk Category"]=="High Risk"
        and customer["Contract"]=="Month-to-Month"
    ):
        return "Promote a long-term contract with exclusive benefits "

    elif(
         customer["Risk Category"] == "Medium Risk"
        and customer["tenure"]<12
    ):
        return "Launch a customer onboarding and engagement campaign."

    elif( customer["TechSupport"] == "No"):
         return "Offer a free technical support trial."

    elif(customer["OnlineSecurity"]=="No"):
        return "Recommend an online security add-on."

    elif( customer["Cluster"] == 0):
        return  "Provide personalized retention offers."

    elif (
        customer["Cluster"] == 2
    ):
        return "Upsell premium internet and entertainment packages."

    else:
        return "Maintain regular customer engagement."


df["Recommendation"]=df.apply(
    recommend,
    axis=1
)

In [12]:
df[
    [
        "customerID",
        "Risk Category",
        "Cluster",
        "Recommendation"
    ]
].head(10)

,customerID,Risk Category,Cluster,Recommendation
0,7590-VHVEG,Medium Risk,0,Launch a customer onboarding and engagement ca...
1,5575-GNVDE,Low Risk,1,Offer a free technical support trial.
2,3668-QPYBK,Medium Risk,1,Launch a customer onboarding and engagement ca...
3,7795-CFOCW,Low Risk,0,Provide personalized retention offers.
4,9237-HQITU,Medium Risk,1,Launch a customer onboarding and engagement ca...
5,9305-CDSKC,High Risk,1,Offer 20% discount and assign a retention spec...
6,1452-KIOVK,Low Risk,1,Offer a free technical support trial.
7,6713-OKOMC,Low Risk,0,Offer a free technical support trial.
8,7892-POOKP,Medium Risk,2,Recommend an online security add-on.
9,6388-TABGU,Low Risk,2,Offer a free technical support trial.


In [13]:
summary = (
    df["Recommendation"]
      .value_counts()
      .reset_index()
)

summary.columns = [
    "Recommendation",
    "Customer Count"
]

summary

,Recommendation,Customer Count
0,Offer a free technical support trial.,2615
1,Maintain regular customer engagement.,1594
2,Recommend an online security add-on.,841
3,Upsell premium internet and entertainment pack...,790
4,Launch a customer onboarding and engagement ca...,768
5,Offer 20% discount and assign a retention spec...,270
6,Provide personalized retention offers.,165


In [14]:
summary.to_csv(
    "../reports/recommendations/recommendation_summary.csv",
    index=False
    
)

In [15]:
df.to_csv(
    "../reports/recommendations/customer_recommendations.csv",
    index=False
)